# 🚀 Фабрика нейросетей & Оптимизатор ансамбля PRL/USDT (Google Colab / GPU T4)

Данный ноутбук реализует исследовательский конвейер глубокого обучения на базе архитектур из `_tools/ltsm_core/model_builder.py` и оптимизатора ансамблей `_tools/ensemble_predictor.py`:

### 🔬 4 специализированные архитектуры + LightGBM:
1. **`conv1d+gru` (Temporal Anchor):** GaussianNoise + LayerNorm + Conv1D Bottleneck + Causal Conv1D + GRU(64) + Dense(64, GELU). Отслеживает динамику OFI, скрытое накопление и айсберг-заявки.
2. **`causal_cnn` (Causal ResNet):** Глубокая остаточная сверточная сеть со строгим `causal`-паддингом и skip-connections. Извлекает геометрию стакана без заглядывания в будущее.
3. **`attention` (Transformer Multi-Head Attention):** 4 головы самовнимания (`key_dim=16`) + Residual Connection + GlobalAveragePooling1D. Улавливает нелокальные временные связи и опережающие импульсы BigONE за 5–15 секунд.
4. **`mlp_bottleneck` (Spatial Dropout Denoiser):** Выбивает `SpatialDropout1D(0.2)` — случайные 20% каналов признаков целиком во времени, защищая от переобучения на доминирующем индикаторе стакана + Residual Dense блоки.

### 🛡️ Защита от переобучения и калибровка вероятностей:
- **Label Smoothing (0.06):** устраняет излишнюю самоуверенность Softmax и калибрует вероятности нейросетей для слияния с резкими порогами деревьев LightGBM.
- **AdamW + Weight Decay (1e-4) + Gradient Clipping (`clipnorm=1.0`):** защита от взрыва градиентов на проливах.
- **Cosine Decay / ReduceLROnPlateau + EarlyStopping:** плавная посадка шага обучения в глобальный минимум.
- **Математический подбор весов (SLSQP + комбинаторный поиск):** находит точные веса альянса $W^* = [w_{\text{lgb}}, w_{\text{gru}}, w_{\text{cnn}}, w_{\text{attn}}, w_{\text{mlp}}]$, максимизирующие OOS IC и минимизирующие Log-Loss.

In [7]:
# 1. Проверка GPU NVIDIA T4 и включение аппаратного ускорения Mixed Precision (FP16)
!nvidia-smi

import os
import gc
import time
import json
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, Conv1D, GaussianNoise,
    GRU, Add, Activation, Flatten, GlobalAveragePooling1D,
    MultiHeadAttention, SpatialDropout1D
)
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.utils.class_weight import compute_class_weight

tf.keras.mixed_precision.set_global_policy('mixed_float16')

print("✅ Версия TensorFlow:", tf.__version__)
print("✅ Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("✅ Политика Mixed Precision:", tf.keras.mixed_precision.global_policy().name)

Sun Oct 18 22:07:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             31W /   70W |   13901MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# 2. Загрузка подготовленного датасета prl_neural_colab_data.npz (27 МБ)
FILE_ID = "1_tX2-62-w4kKJnv9-KFtjmmBo2a1lRt1"
DATA_FILE = "prl_neural_colab_data.npz"

for candidate in [
    "data/prl_neural_colab_data.npz",
    "prl_neural_colab_data.npz",
    "/content/prl_neural_colab_data.npz",
    "/content/drive/MyDrive/prl_neural_colab_data.npz",
    "/content/drive/MyDrive/crypto_bot/prl_neural_colab_data.npz"
]:
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break

if not os.path.exists(DATA_FILE):
    print("☁️ Автоматическое скачивание prl_neural_colab_data.npz с Google Drive...")
    import gdown
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", DATA_FILE, quiet=False)

raw = np.load(DATA_FILE, allow_pickle=True)
X_tr = raw['X_train']
X_te = raw['X_test']

# Строго по закону проекта L = H * K (из train_rolling_swarm.py):
# H=12с -> K=3.5 (Lookback L=42с) | H=5с -> K=3.0 (Lookback L=15с)
HORIZON = "12"
if HORIZON == "12":
    K_RATIO = 3.5
    SEQ_LEN = int(round(12 * K_RATIO))  # 42 секунды!
    y_tr = raw['y_train_h12']
    y_te = raw['y_test_h12']
    dt_te = raw['dt_test_h12']
else:
    K_RATIO = 3.0
    SEQ_LEN = int(round(5 * K_RATIO))   # 15 секунд!
    y_tr = raw['y_train_h5']
    y_te = raw['y_test_h5']
    dt_te = raw['dt_test_h5']

ts_tr = raw['ts_train']
ts_te = raw['ts_test']
feature_names = raw['features']

print(f"✅ Train: {X_tr.shape} ({ts_tr[0]} ➔ {ts_tr[-1]})")
print(f"✅ OOS Test: {X_te.shape} ({ts_te[0]} ➔ {ts_te[-1]})")
print(f"✅ Признаков: {len(feature_names)} | Горизонт H={HORIZON}с | K={K_RATIO} | Lookback L={SEQ_LEN}с")

✅ Train: (210000, 299) (2026-10-15 23:01:18+03:00 ➔ 2026-10-18 09:21:17+03:00)
✅ OOS Test: (20778, 299) (2026-10-18 09:27:30+03:00 ➔ 2026-10-18 15:13:47+03:00)
✅ Признаков: 299 | Горизонт H=12с | K=3.5 | Lookback L=42с


In [9]:
# 3. Нормализация (Zero Data Leakage), веса классов и генераторы 3D-последовательностей
# SEQ_LEN = 42с (для H=12, K=3.5) или 15с (для H=5, K=3.0) задается автоматически в ячейке 2
BATCH_SIZE = 2048  # Безопасный быстрый батч (не переполняет RAM/VRAM)
N_CLASSES = 7

# Статистики строго по Train
mean = np.nanmean(X_tr, axis=0)
std = np.nanstd(X_tr, axis=0) + 1e-6

X_tr_norm = np.clip((X_tr - mean) / std, -5.0, 5.0).astype(np.float32)
X_te_norm = np.clip((X_te - mean) / std, -5.0, 5.0).astype(np.float32)

# One-Hot представление для CategoricalCrossentropy с Label Smoothing
y_tr_oh = tf.keras.utils.to_categorical(y_tr, num_classes=N_CLASSES)
y_te_oh = tf.keras.utils.to_categorical(y_te, num_classes=N_CLASSES)

# Асимметричные веса классов для защиты от ловли падающих ножей
base_weights = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr[SEQ_LEN - 1:])
custom_weights = dict(enumerate(base_weights))
custom_weights[0] *= 1.35  # Класс -3 (сильный пролив)
custom_weights[1] *= 1.25  # Класс -2
custom_weights[2] *= 1.10  # Класс -1

def make_sequence_dataset(X, y_target, seq_len=15, batch_size=512, shuffle=True):
    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=X,
        targets=y_target[seq_len - 1:],
        sequence_length=seq_len,
        sequence_stride=1,
        sampling_rate=1,
        batch_size=batch_size,
        shuffle=shuffle
    )
    return ds.prefetch(tf.data.AUTOTUNE)  # Без .cache(), чтобы не раздувать 3D-окна в 10.5 ГБ RAM!

train_ds = make_sequence_dataset(X_tr_norm, y_tr_oh, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, shuffle=True)
val_ds = make_sequence_dataset(X_te_norm, y_te_oh, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, shuffle=False)

y_true_oos = y_te[SEQ_LEN - 1:]
y_true_oos_oh = y_te_oh[SEQ_LEN - 1:]
dt_true_oos = dt_te[SEQ_LEN - 1:]

print(f"✅ Генераторы готовы! OOS проверочных секунд: {len(y_true_oos):,}")

# Освобождаем исходные ненормализованные массивы для экономии RAM
if "raw" in globals():
    del raw
gc.collect()

✅ Генераторы готовы! OOS проверочных секунд: 20,737


11269

In [10]:
# 4. Фабрика 4 нейросетевых архитектур (из _tools/ltsm_core/model_builder.py)
def build_conv1d_gru(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Conv1D(64, 3, padding='causal', activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.1)(x)
    x = GRU(64, return_sequences=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="conv1d_gru")

def _res_block(x, filters, kernel_size, dilation, l2_reg):
    sc = x
    r = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation, kernel_regularizer=regularizers.l2(l2_reg))(x)
    r = LayerNormalization()(r)
    r = Activation('gelu')(r)
    r = Dropout(0.1)(r)
    r = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation, kernel_regularizer=regularizers.l2(l2_reg))(r)
    r = LayerNormalization()(r)
    if int(sc.shape[-1]) != filters:
        sc = Conv1D(filters, 1, padding='valid', kernel_regularizer=regularizers.l2(l2_reg))(sc)
    return Activation('gelu')(Add()([sc, r]))

def build_causal_cnn(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = _res_block(x, 32, 3, 1, l2_reg)
    x = _res_block(x, 64, 3, 2, l2_reg)
    x = _res_block(x, 64, 3, 4, l2_reg)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="causal_cnn")

def build_attention(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    attn = MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.2)(x, x)
    x = LayerNormalization()(Add()([x, attn]))
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="attention")

def build_mlp_bottleneck(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = SpatialDropout1D(0.20)(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Flatten()(x)
    x = Dense(128, kernel_regularizer=regularizers.l2(l2_reg * 3))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.20)(x)
    sc = x
    x = Dense(128, kernel_regularizer=regularizers.l2(l2_reg * 3))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.20)(x)
    x = Add()([sc, x])
    x = Dense(64, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="mlp_bottleneck")

MODEL_FACTORY = {
    "conv1d+gru": build_conv1d_gru,
    "causal_cnn": build_causal_cnn,
    "attention": build_attention,
    "mlp_bottleneck": build_mlp_bottleneck
}
print("✅ Фабрика из 4 архитектур инициализирована:", list(MODEL_FACTORY.keys()))

✅ Фабрика из 4 архитектур инициализирована: ['conv1d+gru', 'causal_cnn', 'attention', 'mlp_bottleneck']


In [11]:
# 5. Метрики Out-of-Sample и обучение базового эксперта LightGBM (Baseline)
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

if 'X_tr' not in globals() or 'y_true_oos' not in globals():
    raise RuntimeError('❌ Сначала запустите ячейки 1, 2, 3 и 4 сверху вниз (или нажмите Ctrl+F9 / Выполнить все)!')

CLASSES_VEC = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float32)

def evaluate_oos_probs(probs, y_true=y_true_oos, y_oh=y_true_oos_oh, dt_ticks=dt_true_oos):
    p_clip = np.clip(probs, 1e-7, 1.0 - 1e-7)
    p_clip = p_clip / np.sum(p_clip, axis=1, keepdims=True)
    ez = np.dot(p_clip, CLASSES_VEC)
    ic = float(np.corrcoef(ez, dt_ticks)[0, 1]) if np.std(ez) > 1e-6 else 0.0
    try:
        auc = float(roc_auc_score(y_true, p_clip, multi_class='ovr'))
    except Exception:
        auc = 0.5
    ll = float(-np.mean(np.sum(y_oh * np.log(p_clip), axis=1)))
    top2_th = np.quantile(ez, 0.98)
    top2_mask = ez >= top2_th
    top2_pnl = float(np.mean(dt_ticks[top2_mask])) if np.sum(top2_mask) > 0 else 0.0
    return {"ic": ic, "auc": auc, "log_loss": ll, "top2_pnl": top2_pnl}

print("🌲 Обучение референсного эксперта LightGBM на том же срезе...")
t0 = time.time()
dtrain = lgb.Dataset(X_tr, label=y_tr)
lgb_params = {
    'objective': 'multiclass', 'num_class': 7, 'learning_rate': 0.08,
    'num_leaves': 31, 'n_jobs': 4, 'verbose': -1
}
lgb_model = lgb.train(lgb_params, dtrain, num_boost_round=65)
probs_lgb = lgb_model.predict(X_te[SEQ_LEN - 1:])
del dtrain
gc.collect()

oos_predictions = {"lightgbm": probs_lgb}
leaderboard = []
m_lgb = evaluate_oos_probs(probs_lgb)
leaderboard.append({"model": "lightgbm", "params": 0, "time_s": round(time.time() - t0, 1), **m_lgb})
print(f"✅ LightGBM готов ({leaderboard[0]['time_s']}с) | IC: {m_lgb['ic']:+.4f} | AUC: {m_lgb['auc']:.4f} | Loss: {m_lgb['log_loss']:.4f}")

🌲 Обучение референсного эксперта LightGBM на том же срезе...
✅ LightGBM готов (118.4с) | IC: +0.1396 | AUC: 0.7963 | Loss: 0.6132


In [12]:
# 6. Пакетное обучение 4 нейросетевых архитектур с калибровкой (Label Smoothing + AdamW)
ACTIVE_ARCHS = ["conv1d+gru", "causal_cnn", "attention", "mlp_bottleneck"]
EPOCHS = 18
L2_REG = 1e-4

os.makedirs("trained_models", exist_ok=True)
trained_keras_models = {}

for arch in ACTIVE_ARCHS:
    print("\n" + "=" * 70)
    print(f"🚀 ОБУЧЕНИЕ АРХИТЕКТУРЫ: {arch.upper()} (H={HORIZON}с, K={K_RATIO}, Lookback L={SEQ_LEN}с)")
    print("=" * 70)
    tf.keras.backend.clear_session()
    gc.collect()
    t0 = time.time()

    model = MODEL_FACTORY[arch](seq_len=SEQ_LEN, n_features=X_tr.shape[1], n_classes=N_CLASSES, l2_reg=L2_REG)
    optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-2, weight_decay=L2_REG, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.06),
        metrics=['accuracy']
    )

    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, min_delta=1e-4, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
        tf.keras.callbacks.TerminateOnNaN()
    ]

    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=custom_weights, callbacks=cbs, verbose=1)
    elapsed = round(time.time() - t0, 1)

    probs = model.predict(val_ds, verbose=0)
    metrics = evaluate_oos_probs(probs)
    oos_predictions[arch] = probs

    save_path = f"trained_models/model_{arch.replace('+', '_')}_h{HORIZON}s.keras"
    model.save(save_path)
    trained_keras_models[arch] = save_path

    leaderboard.append({"model": arch, "params": model.count_params(), "time_s": elapsed, **metrics})
    print(f"🎯 [{arch.upper()}] OOS IC: {metrics['ic']:+.4f} | ROC-AUC: {metrics['auc']:.4f} | Top-2% PnL: {metrics['top2_pnl']:+.2f} тиков | Loss: {metrics['log_loss']:.4f}")

print("\n🏆 ИТОГОВЫЙ ЛИДЕРБОРД МОДЕЛЕЙ (Out-of-Sample):")
df_lb = pd.DataFrame(leaderboard).sort_values(by="ic", ascending=False)
print(df_lb.to_string(index=False))


🚀 ОБУЧЕНИЕ АРХИТЕКТУРЫ: CONV1D+GRU (H=12с, K=3.5, Lookback L=42с)
Epoch 1/18
103/103 ━━━━━━━━━━━━━━━━━━━━ 39s 315ms/step - accuracy: 0.3130 - loss: 1.5672 - val_accuracy: 0.1566 - val_loss: 2.0460 - learning_rate: 0.0100
Epoch 2/18
103/103 ━━━━━━━━━━━━━━━━━━━━ 41s 321ms/step - accuracy: 0.4617 - loss: 1.1892 - val_accuracy: 0.2287 - val_loss: 2.0763 - learning_rate: 0.0100
Epoch 3/18
103/103 ━━━━━━━━━━━━━━━━━━━━ 36s 325ms/step - accuracy: 0.5107 - loss: 1.0659 - val_accuracy: 0.2402 - val_loss: 2.0228 - learning_rate: 0.0100
Epoch 4/18
103/103 ━━━━━━━━━━━━━━━━━━━━ 40s 318ms/step - accuracy: 0.5557 - loss: 0.9853 - val_accuracy: 0.2448 - val_loss: 2.0392 - learning_rate: 0.0100
Epoch 5/18
102/103 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.5839 - loss: 0.9389
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
103/103 ━━━━━━━━━━━━━━━━━━━━ 42s 324ms/step - accuracy: 0.5849 - loss: 0.9333 - val_accuracy: 0.2810 - val_loss: 2.0445 - learning_rate: 0.0100
Epoch 

In [13]:
# 7. Математический оптимизатор ансамбля (Комбинаторика + SLSQP оптимизация весов)
model_names = list(oos_predictions.keys())
prob_list = [oos_predictions[m] for m in model_names]
M = len(model_names)

print(f"🤝 1. Комбинаторный поиск лучших связок из {M} моделей (по методу ensemble_predictor.py)...")
combo_results = []
for k in range(1, M + 1):
    for indices in itertools.combinations(range(M), k):
        blend = np.mean([prob_list[i] for i in indices], axis=0)
        res = evaluate_oos_probs(blend)
        combo_name = " + ".join([model_names[i] for i in indices])
        combo_results.append({"alliance": combo_name, "k": k, **res})

df_combos = pd.DataFrame(combo_results).sort_values(by=["ic", "auc"], ascending=False)
print("\nТОП-10 РАВНОВЕСНЫХ СВЯЗОК (по OOS IC):")
print(df_combos.head(10).to_string(index=False))

print("\n🧮 2. Непрерывная выпуклая оптимизация весов (Scipy SLSQP: минимизация Log-Loss + максимизация IC)...")
def ensemble_objective(weights):
    w = np.array(weights, dtype=np.float64)
    w = w / np.sum(w)
    blend = sum(w[i] * prob_list[i] for i in range(M))
    p_clip = np.clip(blend, 1e-7, 1.0 - 1e-7)
    ll = -np.mean(np.sum(y_true_oos_oh * np.log(p_clip), axis=1))
    ez = np.dot(p_clip, CLASSES_VEC)
    ic = np.corrcoef(ez, dt_true_oos)[0, 1] if np.std(ez) > 1e-6 else 0.0
    return ll - 0.5 * ic

init_w = np.ones(M) / M
bounds = [(0.0, 1.0) for _ in range(M)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

opt_res = minimize(ensemble_objective, init_w, method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = opt_res.x / np.sum(opt_res.x)

opt_blend = sum(opt_weights[i] * prob_list[i] for i in range(M))
opt_metrics = evaluate_oos_probs(opt_blend)

print("\n" + "=" * 70)
print("🏆 ОПТИМАЛЬНЫЕ ВЕСА МУЛЬТИМОДЕЛЬНОГО АНСАМБЛЯ (W*):")
for name, w in zip(model_names, opt_weights):
    print(f"  • {name:<18}: {w*100:6.2f}%")
print("-" * 70)
print(f"📈 Метрики ансамбля W* vs Одиночный LightGBM:")
print(f"  • OOS IC:          {opt_metrics['ic']:+.4f}  (LightGBM: {m_lgb['ic']:+.4f})")
print(f"  • OOS ROC-AUC:     {opt_metrics['auc']:.4f}  (LightGBM: {m_lgb['auc']:.4f})")
print(f"  • OOS Log-Loss:    {opt_metrics['log_loss']:.4f}  (LightGBM: {m_lgb['log_loss']:.4f})")
print(f"  • Top-2% PnL:      {opt_metrics['top2_pnl']:+.2f} тиков")
print("=" * 70)

🤝 1. Комбинаторный поиск лучших связок из 5 моделей (по методу ensemble_predictor.py)...

ТОП-10 РАВНОВЕСНЫХ СВЯЗОК (по OOS IC):
                                                       alliance  k       ic      auc  log_loss  top2_pnl
            lightgbm + conv1d+gru + causal_cnn + mlp_bottleneck  4 0.216057 0.812295  0.851503  0.086747
                         lightgbm + conv1d+gru + mlp_bottleneck  3 0.208319 0.811785  0.885387  0.098795
                       conv1d+gru + causal_cnn + mlp_bottleneck  3 0.207667 0.807488  1.188936  0.103614
                         lightgbm + causal_cnn + mlp_bottleneck  3 0.204116 0.807764  0.842162  0.077108
                             lightgbm + conv1d+gru + causal_cnn  3 0.203785 0.801542  0.712221  0.113253
lightgbm + conv1d+gru + causal_cnn + attention + mlp_bottleneck  5 0.197812 0.805259  0.894643  0.086747
                                    conv1d+gru + mlp_bottleneck  2 0.197188 0.804659  1.465377  0.113253
                               

In [14]:
# 8. Упаковка всех обученных моделей, паспортов и конфигурации ансамбля для торгового бота
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
os.makedirs(PACK_DIR, exist_ok=True)

for arch, k_path in trained_keras_models.items():
    arch_dir = os.path.join(PACK_DIR, f"{arch.replace('+', '_')}_h{HORIZON}s")
    os.makedirs(arch_dir, exist_ok=True)
    target_keras = os.path.join(arch_dir, "model_win_00.keras")
    import shutil
    shutil.copy2(k_path, target_keras)

    arch_metrics = next(item for item in leaderboard if item["model"] == arch)
    meta = {
        "model_type": arch,
        "coin": "PRL",
        "H": int(HORIZON),
        "seq_len": SEQ_LEN,
        "train_ts_range": [str(ts_tr[0])[:19], str(ts_tr[-1])[:19]],
        "test_ts_range": [str(ts_te[0])[:19], str(ts_te[-1])[:19]],
        "metrics_oos": {
            "ic_dt_ticks": round(arch_metrics["ic"], 4),
            "roc_auc_ovr": round(arch_metrics["auc"], 4),
            "log_loss": round(arch_metrics["log_loss"], 4),
            "top2pct_mean_ticks": round(arch_metrics["top2_pnl"], 2)
        },
        "feature_norm": {"mean": mean.tolist(), "std": std.tolist()},
        "weights_file": "model_win_00.keras"
    }
    with open(os.path.join(arch_dir, "meta_win_00.json"), "w") as f:
        json.dump(meta, f, indent=2)

alliance_cfg = {
    "coin": "PRL",
    "H": int(HORIZON),
    "seq_len": SEQ_LEN,
    "optimal_weights": {name: round(float(w), 4) for name, w in zip(model_names, opt_weights)},
    "ensemble_metrics_oos": {k: round(float(v), 4) for k, v in opt_metrics.items()},
    "leaderboard": leaderboard
}
with open(os.path.join(PACK_DIR, "ensemble_alliance.json"), "w") as f:
    json.dump(alliance_cfg, f, indent=2)

ZIP_NAME = f"{PACK_DIR}.zip"
!zip -r -q {ZIP_NAME} {PACK_DIR}/

print(f"📦 Архив {ZIP_NAME} готов! Ссылки для скачивания в локальный проект:")
print("🚀 Вариант 1 (Catbox Litterbox):")
!curl -s -F "reqtype=fileupload" -F "time=24h" -F "fileToUpload=@{ZIP_NAME}" https://litterbox.catbox.moe/resources/internals/api.php
print("\n🚀 Вариант 2 (tmpfiles.org):")
!curl -s -F "file=@{ZIP_NAME}" https://tmpfiles.org/api/v1/upload

try:
    from google.colab import files
    files.download(ZIP_NAME)
except Exception:
    pass

📦 Архив prl_ensemble_pack_h12s.zip готов! Ссылки для скачивания в локальный проект:
🚀 Вариант 1 (Catbox Litterbox):
https://litter.catbox.moe/bic05v.zip
🚀 Вариант 2 (tmpfiles.org):
{"status":"success","data":{"url":"https://tmpfiles.org/wNADco40uWm7/prl_ensemble_pack_h12s.zip"}}

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>